# 05. Polynomial Terms, Interactions, and Categorical Predictors

A regression model is “linear” when it is linear in the unknown coefficients. It can still contain transformed predictors such as $x^2$, products such as $x_1x_2$, and dummy variables for categories.


In [ ]:
from lite_setup import ensure_packages
await ensure_packages()

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as st
import statsmodels.api as sm
import statsmodels.formula.api as smf

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True


## Curvature with a Quadratic Term

A quadratic model has the form

$$Y=\beta_0+\beta_1x+\beta_2x^2+\epsilon.$$

The model is linear in $\beta_0,\beta_1,\beta_2$, so ordinary least squares still applies.


In [ ]:
mileage = pd.read_csv("data/mileage_additive.csv")
quad = smf.ols("MPG ~ AdditivePercent + I(AdditivePercent**2)", data=mileage).fit()
print(quad.summary())

x_grid = np.linspace(mileage.AdditivePercent.min(), mileage.AdditivePercent.max(), 100)
pred_grid = pd.DataFrame({"AdditivePercent": x_grid})
plt.scatter(mileage.AdditivePercent, mileage.MPG, label="observed")
plt.plot(x_grid, quad.predict(pred_grid), color="black", label="quadratic fit")
plt.xlabel("Additive percent")
plt.ylabel("MPG")
plt.legend()
plt.show()


The hierarchy principle says that if a model includes $x^2$, it should usually also include $x$ unless there is a clear scientific reason not to.

## Interaction Terms

An interaction allows the slope of one predictor to depend on another predictor:

$$Y=\beta_0+\beta_1x_1+\beta_2x_2+\beta_{12}x_1x_2+\epsilon.$$


In `statsmodels` formula syntax, `I(AdditivePercent**2)` tells Python to create the squared predictor before fitting the regression model. This is the Python analogue of explicitly creating a new column for the higher-order term.


In [ ]:
ads = pd.read_csv("data/advertising_sales.csv")
interaction = smf.ols("Sales ~ RadioSpend * PrintSpend", data=ads).fit()
print(interaction.summary())


In formula syntax, `RadioSpend * PrintSpend` expands to `RadioSpend + PrintSpend + RadioSpend:PrintSpend`. With the interaction included, the effect of radio spending is

$$\frac{\partial E[Y]}{\partial RadioSpend}=\beta_1+\beta_{12}PrintSpend.$$


The hierarchy principle also applies to interactions: if `RadioSpend:PrintSpend` is in the model, the corresponding main effects should usually remain in the model too. The shorthand `RadioSpend * PrintSpend` enforces that convention automatically.


In [ ]:
for print_level in [1, 3, 5]:
    radio_slope = interaction.params["RadioSpend"] + interaction.params["RadioSpend:PrintSpend"] * print_level
    print(f"Estimated radio slope when PrintSpend={print_level}: {radio_slope:.3f}")


## Categorical Predictors and Dummy Variables

A categorical variable with $g$ levels is represented with $g-1$ dummy variables when the model has an intercept. The omitted level is the reference category.


Using all $g$ dummy variables together with an intercept would make the design matrix singular. The $g-1$ coding avoids that redundancy and makes every dummy coefficient a comparison against the reference level.


In [ ]:
stores = pd.read_csv("data/store_sales.csv")
cat_model = smf.ols('SalesVolume ~ Households + C(Location, Treatment(reference="Street"))', data=stores).fit()
print(cat_model.summary())


Here, `Street` is the reference category. The `Mall` and `Downtown` coefficients estimate differences from the street-store baseline after accounting for the number of households.


In [ ]:
new_stores = pd.DataFrame({
    "Households": [120, 120, 120],
    "Location": ["Street", "Mall", "Downtown"],
})
new_stores.assign(PredictedSales=cat_model.predict(new_stores))
